In [ ]:
USE DATABASE "FLIGHT_ANALYTICS_DB";
USE SCHEMA "PUBLIC_SCHEMA";

In [ ]:
# Import python packages


# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
LS @FLIGHT_ANALYTICS_DB.PUBLIC.FLIGHT_STAGE;

In [ ]:
SELECT $1
from @FLIGHT_ANALYTICS_DB.PUBLIC.FLIGHT_STAGE
limit 10


In [ ]:
create or replace table "FLIGHT_RAW_NEW" (
    TRANSACTIONID varchar
    ,FLIGHTDATE varchar
    ,AIRLINECODE varchar
    ,AIRLINENAME varchar
    ,TAILNUM varchar
    ,FLIGHTNUM varchar
    ,ORIGINAIRPORTCODE varchar
    ,ORIGAIRPORTNAME varchar
    ,ORIGINCITYNAME varchar
    ,ORIGINSTATE varchar
    ,ORIGINSTATENAME varchar
    ,DESTAIRPORTCODE varchar
    ,DESTAIRPORTNAME varchar
    ,DESTCITYNAME varchar
    ,DESTSTATE varchar
    ,DESTSTATENAME varchar
    ,CRSDEPTIME varchar
    ,DEPTIME varchar
    ,DEPDELAY varchar
    ,TAXIOUT varchar
    ,WHEELSOFF varchar
    ,WHEELSON varchar
    ,TAXIIN varchar
    ,CRSARRTIME varchar
    ,ARRTIME varchar
    ,ARRDELAY varchar
    ,CRSELAPSEDTIME varchar
    ,ACTUALELAPSEDTIME varchar
    ,CANCELLED varchar
    ,DIVERTED varchar
    ,DISTANCE varchar
)


In [ ]:
copy into "FLIGHT_RAW_NEW"
from @FLIGHT_ANALYTICS_DB.PUBLIC.FLIGHT_STAGE
FILE_FORMAT = (TYPE = CSV COMPRESSION = GZIP FIELD_OPTIONALLY_ENCLOSED_BY = '"' FIELD_DELIMITER = '|' SKIP_HEADER = 1)


In [ ]:

CREATE FILE FORMAT IF NOT EXISTS flight_gz
  TYPE = CSV
  FIELD_DELIMITER = '|'
  SKIP_HEADER = 1
  COMPRESSION = GZIP;


CREATE OR REPLACE TABLE flight_raw (data variant);



GARETH EDITS START FROM HERE

In [ ]:
CREATE OR REPLACE TABLE "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" (
    "AIRPORTCODE"     STRING,
    "AIRPORTNAME"    STRING,
    "CITYNAME"   STRING,
    "STATE"   STRING,
    "STATENAME"   STRING
);

INSERT INTO "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT"
with origairport as (
    select distinct
        "ORIGINAIRPORTCODE" as "AIRPORTCODE",
        split_part("ORIGAIRPORTNAME",': ', 2) as "AIRPORTNAME",
        "ORIGINCITYNAME" as "CITYNAME",
        "ORIGINSTATE" as "STATE",
        "ORIGINSTATENAME" as "STATENAME"
    from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
),
destairport as (
    select distinct
        "DESTAIRPORTCODE" as "AIRPORTCODE",
        split_part("DESTAIRPORTNAME",': ', 2) as "AIRPORTNAME",
        "DESTCITYNAME" as "CITYNAME",
        "DESTSTATE" as "STATE",
        "DESTSTATENAME" as "STATENAME"
    from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
)
select * from origairport
union
select * from destairport

In [ ]:
CREATE OR REPLACE TABLE "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE" (
    "AIRLINECODE"     STRING,
    "AIRLINENAME"    STRING
);

INSERT INTO "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE"
SELECT DISTINCT
    "AIRLINECODE",
    SPLIT_PART("AIRLINENAME", ':', 1) as "AIRLINENAME"
FROM "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW";

-- -- CHECK: Should be empty
-- SELECT "AIRLINECODE", COUNT(*) FROM "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE"
-- GROUP BY "AIRLINECODE"
-- HAVING COUNT(*)>1;

In [ ]:
-- DIM_DATE: A bit pointless but keeping to demonstrate.
CREATE OR REPLACE TABLE "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE" (
    "DATE_KEY"            INTEGER,
    "FULL_DATE"           DATE,
    "YEAR"                INTEGER,
    "QUARTER"             INTEGER,
    "MONTH"               INTEGER,
    "MONTH_NAME"          STRING,
    "DAY"                 INTEGER,
    "DAY_OF_WEEK"         INTEGER,
    "DAY_OF_WEEK_NAME"    STRING,
    "WEEK_OF_YEAR"        INTEGER,
    "IS_WEEKEND"          BOOLEAN,
    "PREVIOUS_DAY_KEY"    INTEGER,
    "NEXT_DAY_KEY"        INTEGER
);

SET MIN_DATE = (select MIN(TO_DATE("FLIGHTDATE", 'YYYYMMDD')) from "FLIGHT_RAW_NEW");
SET MAX_DATE = (select MAX(TO_DATE("FLIGHTDATE", 'YYYYMMDD')) from "FLIGHT_RAW_NEW");

INSERT INTO "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE"
with date_span AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY seq4()) - 1 as row_num
    FROM TABLE(
        GENERATOR(
            ROWCOUNT => 366 * (year($MAX_DATE) - year($MIN_DATE) + 1)
            )
        )
    ),
dim_date_columns as (
    SELECT
        DATEADD('day', row_num, datefromparts(year($MIN_DATE), 1, 1)) as d,
        TO_NUMBER(TO_CHAR(d, 'YYYYMMDD')) AS "DATE_KEY",
        YEAR(d) AS "YEAR",
        QUARTER(d) AS "QUARTER",
        MONTH(d) AS "MONTH",
        MONTHNAME(d) AS "MONTH_NAME",
        DAY(d) AS "DAY",
        DAYOFWEEK(d) AS "DAY_OF_WEEK",
        DAYNAME(d) AS "DAY_OF_WEEK_NAME",
        WEEKOFYEAR(d) AS "WEEK_OF_YEAR",
        CASE WHEN DAYOFWEEK(d) IN (6, 7) THEN TRUE ELSE FALSE END AS "IS_WEEKEND",
        TO_NUMBER(TO_CHAR(d - 1, 'YYYYMMDD')) AS "PREVIOUS_DAY_KEY",
        TO_NUMBER(TO_CHAR(d + 1, 'YYYYMMDD')) AS "NEXT_DAY_KEY"
    FROM date_span)
select
    "DATE_KEY",
    d as"FULL_DATE",
    "YEAR",
    "QUARTER",
    "MONTH",
    "MONTH_NAME",
    "DAY",
    "DAY_OF_WEEK",
    "DAY_OF_WEEK_NAME",
    "WEEK_OF_YEAR",
    "IS_WEEKEND",
    "PREVIOUS_DAY_KEY",
    "NEXT_DAY_KEY"
from dim_date_columns
where "YEAR" <= year($MAX_DATE)
;

In [ ]:
CREATE OR REPLACE TABLE "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_BIN" (
    "BIN_KEY"    INTEGER,
    "BIN_NAME"    STRING,
    "MIN_MILES"   INTEGER,
    "MAX_MILES"   INTEGER
);

set MAX_CEIL = 
(    SELECT
        CEIL(MAX(TO_NUMBER(REGEXP_SUBSTR("DISTANCE", '^[0-9]+'))) / 100)
    FROM "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
);

INSERT INTO "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_BIN"
with bins AS (
    SELECT
        SEQ4() AS "BIN_INDEX",
        (SEQ4() * 100) + 1 AS "START_PLUS_ONE",
        (SEQ4() + 1) * 100 AS "END_VALUE",
        0 AS "ZERO_START"
    FROM TABLE(GENERATOR(ROWCOUNT => $MAX_CEIL))
)
SELECT
    "BIN_INDEX" + 1 AS "BIN_KEY",
    CASE 
        WHEN "BIN_INDEX" = 0 THEN '0-100 miles'
        ELSE CONCAT("START_PLUS_ONE", '-', "END_VALUE", ' miles')
    END AS "BIN_NAME",
    CASE 
        WHEN "BIN_INDEX" = 0 THEN 0
        ELSE "START_PLUS_ONE"
    END AS "MIN_MILES",
    CASE 
        WHEN "BIN_INDEX" = 0 THEN 100
        ELSE "END_VALUE"
    END AS "MAX_MILES"
FROM bins;

In [ ]:
CREATE OR REPLACE TABLE "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS" (
    "TRANSACTIONID"     INTEGER,
    "FLIGHTDATE"     INTEGER, 
    "AIRLINECODE"   STRING,
    "TAILNUM"     STRING, 
    "FLIGHTNUM"     INTEGER, 
    "ORIGINAIRPORTCODE"     STRING,
    "DESTAIRPORTCODE"     STRING,
    "CRSDEPTIME"     INTEGER,
    "DEPTIME"     INTEGER,
    "TAXIOUT"     INTEGER,
    "WHEELSOFF"     INTEGER,
    "WHEELSON"     INTEGER,
    "TAXIIN"     INTEGER,
    "CRSARRTIME"     INTEGER,
    "ARRTIME"     INTEGER,
    "ARRDELAY"     INTEGER,
    "CRSELAPSEDTIME"     INTEGER,
    "ACTUALELAPSEDTIME"     INTEGER,
    "CANCELLED"     BOOLEAN,
    "DIVERTED"     BOOLEAN,
    "DEPDELAY"     INTEGER,
    "DEPDELAYGT15"     TINYINT,
    "ARRIVALTIMESTAMP"  TIMESTAMP,
    "NEXTDAYARR"    TINYINT,
    "DISTANCE"     INTEGER,
    "DISTANCEGROUP" STRING
);

INSERT INTO "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS"
SELECT
    "TRANSACTIONID",
    "FLIGHTDATE",
    "AIRLINECODE",
    "TAILNUM", --Not enough time to fix, can talk about in presentation 
    "FLIGHTNUM", 
    "ORIGINAIRPORTCODE",
    "DESTAIRPORTCODE",
    "CRSDEPTIME",
    "DEPTIME",
    "TAXIOUT",
    "WHEELSOFF",
    "WHEELSON",
    "TAXIIN",
    "CRSARRTIME",
    "ARRTIME",
    "ARRDELAY",
    "CRSELAPSEDTIME",
    "ACTUALELAPSEDTIME",
    case 
        when "CANCELLED" in ('0', 'False', 'F') then false
        when "CANCELLED" in ('1', 'True', 'T') then true
    end as "CANCELLED",
    case 
        when "DIVERTED" in ('0', 'False', 'F') then false
        when "DIVERTED" in ('1', 'True', 'T') then true
    end as "DIVERTED",
    "DEPDELAY",
    case 
        when "DEPDELAY" > 15 then 1
        else 0
    end as "DEPDELAYGT15",
    DATEADD(
        minute,
        "ACTUALELAPSEDTIME",
        TO_TIMESTAMP_NTZ(
            CONCAT(
                TO_CHAR(TO_DATE("FLIGHTDATE", 'YYYYMMDD'), 'YYYY-MM-DD'),
                ' ',
                LPAD(IFF("DEPTIME" = '2400', '0000', "DEPTIME"), 4, '0')
            ),
            'YYYY-MM-DD HH24MI')
        ) AS "ARRIVALTIMESTAMP",
    case
        when DATEDIFF(
                day,
                TO_DATE("FLIGHTDATE", 'YYYYMMDD'),
                cast("ARRIVALTIMESTAMP" as date)
             ) = 1
        then 1
        else 0
    end as "NEXTDAYARR",
    TO_NUMBER(REGEXP_SUBSTR("DISTANCE", '^[0-9]+')) AS "DISTANCE",
    bins."BIN_NAME" as "DISTANCEGROUP"
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
-- where "NEXTDAYARR" = 1
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_BIN" bins
on REGEXP_SUBSTR("FLIGHT_RAW_NEW"."DISTANCE", '^[0-9]+') >= bins."MIN_MILES" and REGEXP_SUBSTR("FLIGHT_RAW_NEW"."DISTANCE", '^[0-9]+') <= bins."MAX_MILES";

In [ ]:
/*CREATE OR REPLACE VIEW "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS" AS

select
    flights."TRANSACTIONID",
    flights."DISTANCEGROUP",
    flights."DEPDELAYGT15",
    flights."NEXTDAYARR",
    airline."AIRLINENAME",
    airportorig."AIRPORTNAME" as "ORIGAIRPORTNAME",
    airportdest."AIRPORTNAME" as "DESTAIRPORTNAME",
    flights."CANCELLED",
    flights."DIVERTED"
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS" flights
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" airportorig
on flights."ORIGINAIRPORTCODE" = airportorig."AIRPORTCODE"
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" airportdest
on flights."DESTAIRPORTCODE" = airportdest."AIRPORTCODE"
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE" airline
on flights."AIRLINECODE" = airline."AIRLINECODE"; */

In [ ]:
CREATE OR REPLACE VIEW "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS" AS
WITH base AS (
  SELECT
      flights.*,
      airline."AIRLINENAME",
      airportorig."AIRPORTNAME" AS "ORIGAIRPORTNAME",
      airportdest."AIRPORTNAME" AS "DESTAIRPORTNAME",

      -- Date as real DATE for BI
      TRY_TO_DATE(TO_VARCHAR(flights."FLIGHTDATE"), 'YYYYMMDD') AS "FLIGHTDATE_DT",

      -- Build HH:MM strings safely (for TIME / TIMESTAMP conversion)
      CASE WHEN flights."CRSDEPTIME" IS NULL THEN NULL
           ELSE SUBSTR(LPAD(flights."CRSDEPTIME"::STRING,4,'0'),1,2) || ':' || SUBSTR(LPAD(flights."CRSDEPTIME"::STRING,4,'0'),3,2)
      END AS "CRSDEPTIME_STR",

      CASE WHEN flights."DEPTIME" IS NULL THEN NULL
           ELSE SUBSTR(LPAD(flights."DEPTIME"::STRING,4,'0'),1,2) || ':' || SUBSTR(LPAD(flights."DEPTIME"::STRING,4,'0'),3,2)
      END AS "DEPTIME_STR",

      CASE WHEN flights."CRSARRTIME" IS NULL THEN NULL
           ELSE SUBSTR(LPAD(flights."CRSARRTIME"::STRING,4,'0'),1,2) || ':' || SUBSTR(LPAD(flights."CRSARRTIME"::STRING,4,'0'),3,2)
      END AS "CRSARRTIME_STR",

      CASE WHEN flights."ARRTIME" IS NULL THEN NULL
           ELSE SUBSTR(LPAD(flights."ARRTIME"::STRING,4,'0'),1,2) || ':' || SUBSTR(LPAD(flights."ARRTIME"::STRING,4,'0'),3,2)
      END AS "ARRTIME_STR"

  FROM "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS" flights
  LEFT JOIN "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" airportorig
      ON flights."ORIGINAIRPORTCODE" = airportorig."AIRPORTCODE"
  LEFT JOIN "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" airportdest
      ON flights."DESTAIRPORTCODE" = airportdest."AIRPORTCODE"
  LEFT JOIN "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE" airline
      ON flights."AIRLINECODE" = airline."AIRLINECODE"
)
SELECT
    -- Required columns
    "TRANSACTIONID",
    "DISTANCEGROUP",
    "DEPDELAYGT15",
    "NEXTDAYARR",
    "AIRLINENAME",
    "ORIGAIRPORTNAME",
    "DESTAIRPORTNAME",

    -- Useful extras
    "FLIGHTDATE",
    "FLIGHTDATE_DT",

    "AIRLINECODE",
    "ORIGINAIRPORTCODE",
    "DESTAIRPORTCODE",

    "DISTANCE",
    "DEPDELAY",
    "ARRDELAY",

    "CRSDEPTIME",
    "DEPTIME",
    "CRSARRTIME",
    "ARRTIME",

    "CANCELLED",
    "DIVERTED",

    -- TIME
    TRY_TO_TIME("CRSDEPTIME_STR") AS "CRSDEPTIME_TIME",
    TRY_TO_TIME("DEPTIME_STR")    AS "DEPTIME_TIME",
    TRY_TO_TIME("CRSARRTIME_STR") AS "CRSARRTIME_TIME",
    TRY_TO_TIME("ARRTIME_STR")    AS "ARRTIME_TIME",

    -- TIMESTAMP 
    TRY_TO_TIMESTAMP_NTZ(TO_VARCHAR("FLIGHTDATE_DT") || ' ' || "CRSDEPTIME_STR") AS "SCHED_DEP_TS",
    TRY_TO_TIMESTAMP_NTZ(TO_VARCHAR("FLIGHTDATE_DT") || ' ' || "DEPTIME_STR")    AS "ACTUAL_DEP_TS",

    -- arrival timestamps should respect NEXTDAYARR
    TRY_TO_TIMESTAMP_NTZ(TO_VARCHAR(DATEADD(day, COALESCE("NEXTDAYARR",0), "FLIGHTDATE_DT")) || ' ' || "CRSARRTIME_STR") AS "SCHED_ARR_TS",
    TRY_TO_TIMESTAMP_NTZ(TO_VARCHAR(DATEADD(day, COALESCE("NEXTDAYARR",0), "FLIGHTDATE_DT")) || ' ' || "ARRTIME_STR")    AS "ACTUAL_ARR_TS"

FROM base;

In [ ]:
select * from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS";

SOME DATA QUALITY CHECKS WERE PERFORMED:

- Row count validation between raw, fact, and view tables  
- Verification of distance bin distribution  
- Validation of derived flags (DEPDELAYGT15, NEXTDAYARR)  
- Null analysis for key analytical columns  

All checks confirmed expected results.

In [ ]:
-- Check: Row count comparison between RAW, FACT, and VIEW tables
-- Purpose: Ensure no data was lost during transformations

select
    (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW") as RAW_COUNT,
    (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS") as FACT_COUNT,
    (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS") as VIEW_COUNT;

In [ ]:
-- Check: TRANSACTIONID null validation
-- Purpose: Ensure key identifier is always populated

select
    count(*) as total_rows,
    sum(case when "TRANSACTIONID" is null then 1 else 0 end) as null_transactionids
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS";

In [ ]:

-- Check: DIM_AIRPORT row count
-- Purpose: Verify that airport dimension was populated correctly

select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT";

In [ ]:
-- Check: Preview DIM_AIRPORT data
-- Purpose: Confirm airport names were cleaned and loaded correctly

select *
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT"
limit 10;

In [ ]:
-- Check: DIM_AIRLINE row count
-- Purpose: Verify airline dimension loaded correctly

select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE";

In [ ]:
-- Check: DIM_AIRLINE sample data
-- Purpose: Inspect airline codes and names

select * from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE" limit 10;

In [ ]:
-- Check: DIM_DATE row count
-- Purpose: Verify that the date dimension was populated correctly

select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE";

In [ ]:
-- Check: Validate availability of DIM_BIN and numeric extraction of distance.  
-- Purpose: Map flights into standardized distance bands to support analysis.

select min("FULL_DATE"), max("FULL_DATE")
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE";

In [ ]:
-- Check: DIM_DATE sample records
-- Purpose: Inspect the first few rows to confirm correct date generation and ordering

select *
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE"
order by "FULL_DATE"
limit 10;

In [ ]:
-- Check: Distribution of distance groups
-- Purpose: Validate binning logic and ensure reasonable grouping

select
    "DISTANCEGROUP",
    count(*) as RECORD_COUNT
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS"
group by 1
order by 1;

In [ ]:

-- Check: Preview the final analytical view
-- Purpose: Confirm required columns exist and data looks correct

select *
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS"
limit 10;

In [ ]:
-- Check: DEPDELAYGT15 flag validation
-- Purpose: Ensure derived flag contains only expected values (0 or 1)

select
    "DEPDELAYGT15",
    count(*) as RECORD_COUNT
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS"
group by 1
order by 1;

In [ ]:
-- Check: NEXTDAYARR flag validation
-- Purpose: Ensure derived flag contains only expected values (0 or 1)

select
    "NEXTDAYARR",
    count(*) as RECORD_COUNT
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS"
group by 1
order by 1;

In [ ]:
-- Check: Null value analysis for key analytical columns
-- Purpose: Identify missing values after joins and transformations

select
    sum(case when "TRANSACTIONID" is null then 1 else 0 end) as NULL_TRANSACTIONID,
    sum(case when "DISTANCEGROUP" is null then 1 else 0 end) as NULL_DISTANCEGROUP,
    sum(case when "AIRLINENAME" is null then 1 else 0 end) as NULL_AIRLINENAME,
    sum(case when "ORIGAIRPORTNAME" is null then 1 else 0 end) as NULL_ORIGAIRPORTNAME,
    sum(case when "DESTAIRPORTNAME" is null then 1 else 0 end) as NULL_DESTAIRPORTNAME
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."VW_FLIGHTS";

In [ ]:
select
    count(*) as total_rows,
    count(distinct "TRANSACTIONID") as distinct_transactionids
from FLIGHT_ANALYTICS_DB.PUBLIC_SCHEMA.FACT_FLIGHTS;

In [ ]:
select current_schema();

In [ ]:
show tables in schema FLIGHT_ANALYTICS_DB.PUBLIC_SCHEMA;


In [ ]:
show views  in schema FLIGHT_ANALYTICS_DB.PUBLIC_SCHEMA;

In [ ]:
-- Check: Delay value distribution and anomalies
-- Purpose: Detect negative or unusually large delay values

select
  count(*) as TOTAL_ROWS,
  sum(case when "DEPDELAY" < 0 then 1 else 0 end) as NEG_DEPDELAY,
  sum(case when "ARRDELAY" < 0 then 1 else 0 end) as NEG_ARRDELAY,
  max("DEPDELAY") as MAX_DEPDELAY,
  max("ARRDELAY") as MAX_ARRDELAY
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW";

In [ ]:
-- Check: Time field format validation (HHMM)
-- Purpose: Identify invalid or non-numeric departure and arrival times

with t as (
  select
    "DEPTIME",
    "ARRTIME",
    try_to_number("DEPTIME") as DEP_NUM,
    try_to_number("ARRTIME") as ARR_NUM
  from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
)

select
  count(*) as TOTAL_ROWS,
  sum(case when DEP_NUM is null and "DEPTIME" is not null then 1 else 0 end) as NON_NUMERIC_DEPTIME,
  sum(case when ARR_NUM is null and "ARRTIME" is not null then 1 else 0 end) as NON_NUMERIC_ARRTIME,
  sum(case when DEP_NUM is not null and (floor(DEP_NUM/100) > 23 or mod(DEP_NUM,100) > 59) then 1 else 0 end) as INVALID_DEPTIME,
  sum(case when ARR_NUM is not null and (floor(ARR_NUM/100) > 23 or mod(ARR_NUM,100) > 59) then 1 else 0 end) as INVALID_ARRTIME
from t;

In [ ]:
-- Check: Distance value validation (robust numeric parsing)
-- Purpose: Identify null, zero, or negative distance values even when stored as text

with d as (
  select
    "DISTANCE" as DISTANCE_RAW,
    try_to_number(regexp_replace("DISTANCE"::string, '[^0-9.-]', '')) as DISTANCE_NUM
  from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FLIGHT_RAW_NEW"
)
select
  count(*) as TOTAL_ROWS,
  sum(case when DISTANCE_RAW is null then 1 else 0 end) as NULL_DISTANCE_RAW,
  sum(case when DISTANCE_RAW is not null and DISTANCE_NUM is null then 1 else 0 end) as NON_NUMERIC_DISTANCE,
  sum(case when DISTANCE_NUM < 0 then 1 else 0 end) as NEG_DISTANCE,
  sum(case when DISTANCE_NUM = 0 then 1 else 0 end) as ZERO_DISTANCE,
  min(DISTANCE_NUM) as MIN_DISTANCE,
  max(DISTANCE_NUM) as MAX_DISTANCE
from d;

In [ ]:
-- Check: Dimension join coverage
-- Purpose: Ensure all fact records have matching dimension keys

select
  sum(case when a."AIRLINECODE" is null then 1 else 0 end) as MISSING_AIRLINE,
  sum(case when o."AIRPORTCODE" is null then 1 else 0 end) as MISSING_ORIGIN_AIRPORT,
  sum(case when d."AIRPORTCODE" is null then 1 else 0 end) as MISSING_DEST_AIRPORT
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS" f
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE" a
  on f."AIRLINECODE" = a."AIRLINECODE"
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" o
  on f."ORIGINAIRPORTCODE" = o."AIRPORTCODE"
left join "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT" d
  on f."DESTAIRPORTCODE" = d."AIRPORTCODE";

In [ ]:
-- Check: NEXTDAYARR distribution
-- Purpose: Validate next-day arrival logic distribution

select
  "NEXTDAYARR",
  count(*) as FLIGHTS
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."FACT_FLIGHTS"
group by 1
order by 1;

In [ ]:
-- Check: Dimension table row counts
-- Purpose: Confirm dimension tables were populated

select
  (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE") as DIM_AIRLINE_ROWS,
  (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT") as DIM_AIRPORT_ROWS,
  (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_DATE") as DIM_DATE_ROWS,
  (select count(*) from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_BIN") as DIM_BIN_ROWS;

In [ ]:
-- Check: AIRLINENAME cleanup validation
-- Purpose: Ensure airline code prefix was removed from AIRLINENAME

select *
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRLINE"
where "AIRLINENAME" like '%:%'
limit 20;

In [ ]:
-- Check: AIRPORTNAME cleanup validation
-- Purpose: Ensure AIRPORTNAME no longer contains concatenated city/state

select *
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT"
where "AIRPORTNAME" like '%:%' or "AIRPORTNAME" like '%,%'
limit 20;

In [ ]:
-- Check: Missing airport location attributes
-- Purpose: Identify airport rows with missing city/state fields

select
  sum(case when "CITYNAME" is null then 1 else 0 end) as NULL_CITYNAME,
  sum(case when "STATE" is null then 1 else 0 end) as NULL_STATE,
  sum(case when "STATENAME" is null then 1 else 0 end) as NULL_STATENAME
from "FLIGHT_ANALYTICS_DB"."PUBLIC_SCHEMA"."DIM_AIRPORT";